In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc networkx numpy qiskit-ibm-catalog sympy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     channel="ibm_quantum_platform",
#     token="<your-api-key>",
#     # instance="<IBM Cloud CRN or instance name>",  # optional
#     set_as_default=True,
#     overwrite=True,
# )

# Optimization Solver: O Funcție Qiskit de Q-CTRL Fire Opal
*Consultă [referința API](https://docs.quantum.ibm.com/api/functions/q-ctrl-optimization-solver)*

> **Note:** Funcțiile Qiskit sunt o funcționalitate experimentală disponibilă doar utilizatorilor IBM Quantum&reg; cu abonament Premium Plan, Flex Plan și On-Prem (prin IBM Quantum Platform API) Plan. Acestea se află în stadiu de previzualizare și pot fi modificate.


<Accordion>
<AccordionItem title="Package versions">

The code on this page was developed using the following requirements.
We recommend using these versions or newer.

```
qiskit-ibm-runtime~=0.46.1
sympy~=1.14.0
```
</AccordionItem>
</Accordion>
## Prezentare generală
Cu Fire Opal Optimization Solver, poți rezolva probleme de optimizare la scară utilă pe hardware cuantic fără a necesita expertiză în domeniul cuantic. Pur și simplu introduci definiția problemei la nivel înalt, iar Solver-ul se ocupă de restul. Întregul flux de lucru este conștient de zgomot și utilizează [Fire Opal's Performance Management](/guides/q-ctrl-performance-management) în fundal. Solver-ul oferă în mod constant soluții precise pentru probleme dificil de rezolvat clasic, chiar și la scara completă a dispozitivului, pe cele mai mari QPU-uri IBM&reg;.

Solver-ul este flexibil și poate fi utilizat pentru a rezolva probleme de optimizare combinatorie definite ca funcții obiectiv sau grafuri arbitrare. Problemele nu trebuie să fie mapate la topologia dispozitivului. Atât problemele neconstrânse, cât și cele constrânse pot fi rezolvate, cu condiția că restricțiile pot fi formulate ca termeni de penalizare. Exemplele incluse în acest ghid demonstrează cum să rezolvi o problemă de optimizare neconstrânsă și una constrânsă la scară utilă, folosind diferite tipuri de intrări ale Solver-ului. Primul exemplu implică o problemă max-cut definită pe un graf 3-regulat cu 156 de noduri, în timp ce al doilea exemplu abordează o problemă de Acoperire Minimă a Vârfurilor cu 50 de noduri, definită printr-o funcție cost.

Pentru a obține acces la Optimization Solver, [contactează Q-CTRL](https://form.typeform.com/to/uOAVDnGg?typeform-source=q-ctrl.com).
## Descrierea funcției
Solver-ul optimizează și automatizează complet întregul algoritm, de la suprimarea erorilor la nivel hardware până la maparea eficientă a problemei și optimizarea clasică în buclă închisă. În fundal, pipeline-ul Solver-ului reduce erorile la fiecare etapă, permițând performanța îmbunătățită necesară pentru o scalare semnificativă. Fluxul de lucru de bază este inspirat de Quantum Approximate Optimization Algorithm (QAOA), care este un algoritm hibrid cuantic-clasic. Pentru un rezumat detaliat al fluxului complet de lucru al Optimization Solver, consultă [manuscrisul publicat](https://arxiv.org/abs/2406.01743).

![Vizualizarea fluxului de lucru al Optimization Solver](../docs/images/guides/qctrl-optimization/solver_workflow.svg)

Pentru a rezolva o problemă generică cu Optimization Solver:
1. Definește problema ta ca o funcție obiectiv, un graf sau un lanț de spin `SparsePauliOp`.
2. Conectează-te la funcție prin Catalogul de Funcții Qiskit.
3. Rulează problema cu Solver-ul și recuperează rezultatele.
### Formate de problemă acceptate
- Reprezentarea expresiei polinomiale a unei funcții obiectiv. Ideal creată în Python cu un obiect SymPy Poly existent și formatată într-un șir folosind [sympy.srepr](https://docs.sympy.org/latest/tutorials/intro-tutorial/printing.html#srepr).
- Reprezentarea grafului unui tip specific de problemă. Graful trebuie creat folosind biblioteca networkx în Python. Apoi trebuie convertit într-un șir folosind funcția networkx `[nx.readwrite.json_graph.adjacency_data](http://nx.readwrite.json_graph.adjacency_data.)`.
- Reprezentarea lanțului de spin a unei probleme specifice. Lanțul de spin trebuie reprezentat ca obiect `SparsePauliOp`; consultă [documentația](https://docs.quantum.ibm.com/api/qiskit/qiskit.quantum_info.SparsePauliOp) pentru mai multe detalii.

> **Note:** Dacă vrei să folosești un Backend pe care această funcție nu îl suportă în prezent, [contactează Q-CTRL](https://form.typeform.com/to/iuujEAEI?typeform-source=q-ctrl.com) pentru a adăuga suport.
## Benchmark-uri
[Rezultatele de benchmark publicate](https://arxiv.org/abs/2406.01743) arată că Solver-ul rezolvă cu succes probleme cu peste 120 de Qubiți, depășind chiar rezultatele publicate anterior pe dispozitivele de tip quantum annealing și trapped-ion. Următoarele metrici de benchmark oferă o indicație aproximativă a acurateței și scalabilității tipurilor de probleme, bazate pe câteva exemple. Metricile reale pot diferi în funcție de diverse caracteristici ale problemei, cum ar fi numărul de termeni din funcția obiectiv (densitate) și localitatea lor, numărul de variabile și ordinul polinomului.

„Numărul de Qubiți" indicat nu este o limitare strictă, ci reprezintă praguri aproximative unde poți anticipa o acuratețe a soluției extrem de consistentă. Dimensiunile mai mari ale problemelor au fost rezolvate cu succes, iar testarea dincolo de aceste limite este încurajată.

Conectivitatea arbitrară a Qubiților este suportată pentru toate tipurile de probleme.

| Tip de problemă    | Număr de Qubiți | Exemplu | Acuratețe | Timp total (s) | Utilizare Runtime (s) | Număr de iterații
| ---------  | ---------------- | -------------------------- | -------- | ---------- | ------------- |---- |
| Probleme pătratice cu conectivitate redusă  | 156 | max-cut 3-regulat| 100%     | 1764     | 293          | 16 |
| Optimizare binară de ordin superior | 156 | Model Ising spin-glass | 100%      | 1461     | 272          | 16 |
| Probleme pătratice cu conectivitate densă | 50 | max-cut complet conectat| 100%      |  1758    | 268  | 12 |
| Problemă constrânsă cu termeni de penalizare | 50 | Acoperire Minimă Ponderată a Vârfurilor cu densitate de 8% a muchiilor | 100%      | 1074     | 215 | 10 |
## Începe
Mai întâi, autentifică-te folosind [cheia API IBM Quantum](http://quantum.cloud.ibm.com/). Apoi, selectează Funcția Qiskit după cum urmează. (Acest fragment de cod presupune că ai [salvat deja contul](/guides/functions#install-qiskit-functions-catalog-client) în mediul tău local.)

In [4]:
from qiskit_ibm_catalog import QiskitFunctionsCatalog

catalog = QiskitFunctionsCatalog(channel="ibm_quantum_platform")

# Verify that you have access to the function
catalog.list()

[QiskitFunction(qunova/hivqe-chemistry),
 QiskitFunction(global-data-quantum/quantum-portfolio-optimizer),
 QiskitFunction(algorithmiq/tem),
 QiskitFunction(qedma/qesem),
 QiskitFunction(multiverse/singularity),
 QiskitFunction(ibm/circuit-function),
 QiskitFunction(q-ctrl/optimization-solver),
 QiskitFunction(colibritd/quick-pde),
 QiskitFunction(q-ctrl/performance-management),
 QiskitFunction(kipu-quantum/iskay-quantum-optimizer)]

In [2]:
# Access Function
solver = catalog.load("q-ctrl/optimization-solver")

## Exemplu: Optimizare neconstrânsă
Rulează problema [tăieturii maxime](https://en.wikipedia.org/wiki/Maximum_cut) (Max-Cut). Următorul exemplu demonstrează capacitățile Solver-ului pe o problemă Max-Cut pe un graf 3-regulat neponderat cu 156 de noduri, dar poți rezolva și probleme pe grafuri ponderate.
Pe lângă `qiskit-ibm-catalog`, vei folosi și următoarele pachete pentru a rula acest exemplu: `networkx` și `numpy`. Poți instala aceste pachete decomentând celula de mai jos dacă rulezi acest exemplu într-un notebook folosind kernelul IPython.

In [3]:
# %pip install networkx numpy

### 1. Definește problema
Poți rula o problemă Max-Cut definind o problemă pe bază de graf și specificând `problem_type='maxcut'`.

In [1]:
import networkx as nx
import numpy as np

# Generate a random graph with 156 nodes
maxcut_graph = nx.random_regular_graph(d=3, n=156, seed=8)

In [2]:
# Optionally, visualize the graph
nx.draw_networkx(
    maxcut_graph, nx.kamada_kawai_layout(maxcut_graph), node_size=100
)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.avif" alt="Output of the previous code cell" />

![Rezultatul celulei de cod anterioare](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/0a7255e1-0.svg)

Solver-ul acceptă un șir ca intrare pentru definiția problemei.

In [3]:
# Convert graph to string
problem_as_str = nx.readwrite.json_graph.adjacency_data(maxcut_graph)

### 2. Rulează problema
Când folosești metoda de intrare bazată pe graf, specifică tipul problemei.

In [4]:
# This cell is hidden from users
from qiskit_ibm_runtime import QiskitRuntimeService

service = QiskitRuntimeService()
backend_name = service.least_busy(n_qubits=156).name

In [ ]:
# Solve the problem
maxcut_job = solver.run(
    problem=problem_as_str,
    problem_type="maxcut",
    backend_name=backend_name,  # E.g. "ibm_fez"
)

Verifică [starea](/guides/functions#check-job-status) sarcinii din Funcția Qiskit sau recuperează [rezultatele](/guides/functions#retrieve-results) după cum urmează:

In [9]:
# Print the ID so you can use it later, if necessary
print(maxcut_job.job_id)

# Get job status
print(maxcut_job.status())

34b53970-d95a-4e24-8763-fc6f3d112843


QUEUED


### 3. Retrieve the result
Retrieve the optimal cut value from the results dictionary.

<Admonition type="note">
   The mapping of the variables to the bitstring may have changed. The output dictionary contains a `variables_to_bitstring_index_map` sub-dictionary, which helps to verify the ordering.
</Admonition>

In [10]:
# Poll for results
maxcut_result = maxcut_job.result()

# Take the absolute value of the solution since the cost function is minimized
qctrl_maxcut = abs(maxcut_result["solution_bitstring_cost"])

# Print the optimal cut value found by the Optimization Solver
print(f"Optimal cut value: {qctrl_maxcut}")

Optimal cut value: 210.0


### 3. Recuperează rezultatul
Recuperează valoarea tăieturii optime din dicționarul de rezultate.

> **Note:** Maparea variabilelor la șirul de biți poate fi modificată. Dicționarul de ieșire conține un sub-dicționar `variables_to_bitstring_index_map`, care ajută la verificarea ordinii.

In [11]:
# %pip install numpy networkx sympy

### 1. Define the problem
Define a random graph partitioning problem by generating a graph with randomly weighted nodes.

In [26]:
import networkx as nx
from sympy import Symbol, Poly, srepr

# To change the weights, change the seed to any integer.
rng_seed = 18
_rng = np.random.default_rng(rng_seed)
node_count = 50
edge_probability = 0.08
graph = nx.erdos_renyi_graph(
    node_count, edge_probability, seed=rng_seed, directed=False
)

# add node weights
min_weight = -1.0
max_weight = 1.0
for i in graph.nodes:
    weight = (max_weight - min_weight) * _rng.random() + min_weight
    graph.add_node(i, weight=weight)

# Optionally, visualize the graph
nx.draw_networkx(graph, nx.kamada_kawai_layout(graph), node_size=200)

<Image src="../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.avif" alt="Output of the previous code cell" />

Poți verifica acuratețea rezultatului rezolvând problema clasic cu solvere open-source precum [PuLP](https://coin-or.github.io/pulp/), dacă graful nu este dens conectat. Problemele cu densitate ridicată pot necesita solvere clasice avansate pentru a valida soluția.
## Exemplu: Optimizare constrânsă
Exemplul anterior max-cut este o problemă de optimizare binară pătratică neconstrânsă frecventă. Optimization Solver de la Q-CTRL poate fi utilizat pentru diverse tipuri de probleme, inclusiv optimizare constrânsă. Poți rezolva tipuri arbitrare de probleme introducând definiția problemei reprezentată ca un polinom, unde restricțiile sunt modelate ca termeni de penalizare.

Următorul exemplu demonstrează cum să construiești o funcție cost pentru o problemă de optimizare constrânsă, [acoperirea minimă a vârfurilor](https://en.wikipedia.org/wiki/Vertex_cover) (MVC).
Pe lângă pachetele `qiskit-ibm-catalog` și `qiskit`, vei folosi și următoarele pachete pentru a rula acest exemplu: `numpy`, `networkx` și `sympy`. Poți instala aceste pachete decomentând celula de mai jos dacă rulezi acest exemplu într-un notebook folosind kernelul IPython.

In [27]:
# Construct the cost function.
group_count = 3
variables = [
    Symbol(f"n[{i},{g}]")
    for i in range(node_count)
    for g in range(group_count)
]
node_group_var = {
    (i, g): variables[i * group_count + g]
    for i in range(node_count)
    for g in range(group_count)
}
cost_function = Poly(0, *variables)

for i, j in graph.edges():
    edge_weight = graph.nodes[i]["weight"] + graph.nodes[j]["weight"]
    for g in range(group_count):
        cost_function += (
            edge_weight * node_group_var[(i, g)] * node_group_var[(j, g)]
        )

### 1. Definește problema
Definește o problemă MVC aleatoare generând un graf cu noduri ponderate aleator.

In [28]:
# Build the hard constraint: exactly one group per node.
constraint_dict = {
    str(tuple(f"n[{i},{g}]" for g in range(group_count))): 1
    for i in range(node_count)
}
print(f"Problem constraints: {constraint_dict}")

Problem constraints: {"('n[0,0]', 'n[0,1]', 'n[0,2]')": 1, "('n[1,0]', 'n[1,1]', 'n[1,2]')": 1, "('n[2,0]', 'n[2,1]', 'n[2,2]')": 1, "('n[3,0]', 'n[3,1]', 'n[3,2]')": 1, "('n[4,0]', 'n[4,1]', 'n[4,2]')": 1, "('n[5,0]', 'n[5,1]', 'n[5,2]')": 1, "('n[6,0]', 'n[6,1]', 'n[6,2]')": 1, "('n[7,0]', 'n[7,1]', 'n[7,2]')": 1, "('n[8,0]', 'n[8,1]', 'n[8,2]')": 1, "('n[9,0]', 'n[9,1]', 'n[9,2]')": 1, "('n[10,0]', 'n[10,1]', 'n[10,2]')": 1, "('n[11,0]', 'n[11,1]', 'n[11,2]')": 1, "('n[12,0]', 'n[12,1]', 'n[12,2]')": 1, "('n[13,0]', 'n[13,1]', 'n[13,2]')": 1, "('n[14,0]', 'n[14,1]', 'n[14,2]')": 1, "('n[15,0]', 'n[15,1]', 'n[15,2]')": 1, "('n[16,0]', 'n[16,1]', 'n[16,2]')": 1, "('n[17,0]', 'n[17,1]', 'n[17,2]')": 1, "('n[18,0]', 'n[18,1]', 'n[18,2]')": 1, "('n[19,0]', 'n[19,1]', 'n[19,2]')": 1, "('n[20,0]', 'n[20,1]', 'n[20,2]')": 1, "('n[21,0]', 'n[21,1]', 'n[21,2]')": 1, "('n[22,0]', 'n[22,1]', 'n[22,2]')": 1, "('n[23,0]', 'n[23,1]', 'n[23,2]')": 1, "('n[24,0]', 'n[24,1]', 'n[24,2]')": 1, "('n[25,

![Rezultatul celulei de cod anterioare](../docs/images/guides/q-ctrl-optimization-solver/extracted-outputs/c2ce65e3-0.svg)

Un model de optimizare standard pentru MVC ponderat poate fi formulat după cum urmează. Mai întâi, trebuie adăugată o penalizare pentru orice caz în care o muchie nu este conectată la un vârf din submulțime. Prin urmare, fie $n_i = 1$ dacă vârful $i$ se află în acoperire (adică în submulțime) și $n_i = 0$ altfel. În al doilea rând, scopul este de a minimiza numărul total de vârfuri din submulțime, ceea ce poate fi reprezentat prin următoarea funcție:

$$\textbf{Minimize}\qquad y = \sum_{i\in V} \omega_i n_i$$

In [20]:
# Solve the problem
partition_job = solver.run(
    problem=srepr(cost_function),
    constraint=constraint_dict,
    backend_name="ibm_marrakesh",  # E.g. "ibm_marrakesh"
)

Acum, fiecare muchie din graf trebuie să includă cel puțin un capăt din acoperire, ceea ce poate fi exprimat ca inegalitate:

$$n_i + n_j \ge 1 \texttt{ for all } (i,j)\in E$$

Orice caz în care o muchie nu este conectată la vârful acoperirii trebuie penalizat. Aceasta poate fi reprezentată în funcția cost prin adăugarea unei penalizări de forma $P(1-n_i-n_j+n_i n_j)$ unde $P$ este o constantă de penalizare pozitivă. Astfel, o alternativă neconstrânsă la inegalitatea constrânsă pentru MVC ponderat este:

$$\textbf{Minimize}\qquad y = \sum_{i\in V}\omega_i n_i + P(\sum_{(i,j)\in E}(1 - n_i - n_j + n_i n_j))$$

In [21]:
# Print the ID so you can use it later, if necessary
print(partition_job.job_id)

# Get job status
print(partition_job.status())

b8085944-f313-444e-be39-ea61b1b47ebd
QUEUED


### 2. Rulează problema

In [ ]:
partition_result = partition_job.result()
qctrl_cost = partition_result["solution_bitstring_cost"]
solution_bitstring = partition_result["solution_bitstring"]

# Print results
print(f"Total weight of same-group edges: {qctrl_cost}")
print(f"Solution bitstring: {solution_bitstring}")

Total weight of same-group edges: -36.5539
Solution bitstring: 100100100100100001100100100100100100100100100100100001010100010100100100100010001001100100100001100001100001010001001010100100100100100010100100100100


Verifică [starea](/guides/functions#check-job-status) sarcinii din Funcția Qiskit sau recuperează [rezultatele](/guides/functions#retrieve-results) după cum urmează: